# Data Governance


*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

## What is data governance and why does it matter?

Data governance is the set of processes, policies, and responsibilities that ensure data is discoverable, trusted, secure, and used correctly. It matters because data platforms scale to many teams; without governance, definitions diverge, access becomes risky, and compliance issues appear. Good governance enables faster, safer self-service analytics.

The four pillars interviewers expect:

```text
1. Ownership      - every dataset has an accountable owner and a steward.
2. Discovery      - catalog, glossary, lineage, and clear access request paths.
3. Protection     - classification, least privilege, masking, encryption, audit.
4. Lifecycle      - quality SLAs, retention, archival, and safe deprecation.
```

A governance program is only real when it is enforced in the platform, not just documented. Example enforcement points: catalog tags drive access policies, data contracts are checked in CI, retention is a scheduled job, and access reviews are tracked tickets.

**The policy lifecycle.** Governance runs as a loop: define, enforce, monitor, review. Define writes the rule and names an owner. Enforce puts the rule into the platform rather than into a wiki. Monitor measures whether the rule holds. Review re-approves or retires the rule on a schedule, because data, laws, and owners change.

Worked example, a new `customer_email` column appears:
1. Define: the steward tags it `pii` and the owner sets 30-day raw retention.
2. Enforce: the engine attaches a masking policy and a retention job runs daily.
3. Monitor: a weekly report lists unclassified columns and failed retention runs.
4. Review: the reviewer re-confirms the tag each quarter, or moves the column to a stricter tier.

**Common mistake.** Treating governance as a documentation project. A catalog entry that says "PII, restricted" changes nothing if the engine still returns the raw email. The honest test: can an ordinary analyst read the protected value right now?

A rule is easier to enforce when it is stored as data, because both CI and the catalog can read the same record.

```json
{
  "policy_id": "retention.raw_events",
  "owner": "platform-data@example.com",
  "scope": {"catalog": "main", "schema": "bronze", "table": "events_raw"},
  "rule": {"type": "retention", "days": 90, "action": "delete"},
  "evidence": ["retention_run_log", "catalog_certification"],
  "review_by": "2027-01-15"
}
```

Every field above answers a review question: who is accountable, what exactly is covered, what the rule does, and where the proof lives.

## What is a data contract?

A data contract is an agreement between data producers and consumers about schema, semantics, quality expectations, ownership, and change process. It prevents upstream teams from changing fields silently and breaking downstream pipelines.

Example contract items:

+ `order_id` is required and unique per order
+ `amount` is decimal and must be non-negative
+ `order_status` allowed values are `created`, `paid`, `cancelled`, `refunded`
+ breaking changes require versioning and consumer notice

In interviews, connect contracts to CI checks, schema registries, catalog ownership, and incident reduction.

A contract as code, validated before deployment:

```yaml
# contracts/orders_v1.yaml
dataset: gold.fact_orders
owner: orders-platform@example.com
stewards: [data-platform@example.com]
sla:
  freshness_minutes: 30
  availability: "99.9%"
schema:
  - name: order_id
    type: string
    required: true
    unique: true
  - name: amount
    type: decimal(12,2)
    required: true
    constraints: ["amount >= 0"]
  - name: order_status
    type: string
    enum: [created, paid, cancelled, refunded]
    required: true
quality_checks:
  - name: no_null_order_id
    sql: "select count(*) from {table} where order_id is null"
    expect: 0
change_policy:
  breaking: requires_new_version_and_consumer_signoff
  additive: allowed_with_notice
consumers:
  - name: finance_marts
    contact: finance-data@example.com
```

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
# CI check: fail the build if the produced table violates the contract.
from pyspark.sql import functions as F

contract_violations = (
    spark.table("gold.fact_orders")
    .select(
        F.sum(F.col("order_id").isNull().cast("int")).alias("null_ids"),
        F.sum((F.col("amount") < 0).cast("int")).alias("negative_amounts"),
        F.sum((~F.col("order_status").isin("created", "paid", "cancelled", "refunded")).cast("int")).alias("bad_status"),
    )
    .first()
)

assert contract_violations.null_ids == 0, "order_id contract violated"

**How enforcement works.** A contract is only a contract when a check can fail. The producer publishes a contract version beside the table. A CI job compares the produced schema and the declared expectations against that version and blocks the deploy when they disagree. The registry and the catalog then expose the contract so consumers can see what they depend on.

**Change types.** Adding a nullable column is usually backward compatible. Renaming a column, narrowing a type, or making a nullable column required breaks existing readers. The contract should state its compatibility rule, for example backward compatible within a major version.

Worked example: a producer renames `amount` to `order_amount` in `gold.fact_orders`. The contract check reads the declared schema, fails with a diff, and the producer either keeps `amount` for one deprecation window or bumps the contract to v2 and coordinates with the consumers that lineage finds.

```json
{
  "contract_id": "orders_v1",
  "dataset": "gold.fact_orders",
  "owner": "orders-platform@example.com",
  "schema": [
    {"name": "order_id", "type": "string", "required": true},
    {"name": "amount", "type": "decimal(18,2)", "required": true, "min": 0}
  ],
  "compatibility": "backward",
  "sla": {"freshness_hours": 6}
}
```

**Common mistake.** Keeping the contract in a wiki and the schema in the pipeline. The two drift, and the drift is discovered only when a dashboard breaks.

## How do lineage and cataloging help governance?

Cataloging tells users what datasets exist, who owns them, what fields mean, and how to request access. Lineage shows where data came from and what downstream assets depend on it. Together they help with impact analysis, audits, and trust.

Example: before dropping `customer_email`, lineage can show dashboards, dbt models, and ML features that still use the column. The owner can then coordinate a safe deprecation instead of causing a production incident.

Lineage has two useful granularities:

+ Table-level lineage: fast impact analysis for schema changes and outages.
+ Column-level lineage: needed for PII propagation, because it answers "which derived columns contain data from `customer_email`?"

```sql
-- Query Unity Catalog table lineage (Delta sharing style metadata).
select
  source_table_full_name,
  target_table_full_name,
  event_time
from system.access.table_lineage
where target_table_full_name = 'main.gold.fact_orders'
  and event_time > current_timestamp() - interval 30 days
order by event_time desc;
```

Governance use cases for lineage: PII propagation analysis, cost attribution, deprecation impact, incident blast radius, and proving in an audit which systems processed regulated data.

**How catalog metadata is harvested.** A crawler connects to the warehouse or lake, reads the information schema on a schedule, and writes tables, columns, types, comments, tags, and owners into the catalog. Push-based systems emit an event when a job writes data, which is how OpenLineage-style lineage stays close to real time. Pull-based crawlers are simpler and cheaper but always lag the platform.

Lineage also has two directions. Upstream lineage answers "where did this number come from", which is the question during an incident. Downstream lineage answers "what breaks if I change this", which is the question during a schema change.

Worked example: you plan to drop `customers.email`. Column-level lineage names the exact downstream columns that read it.

```sql
-- Databricks: downstream columns that read a source column.
select distinct
  target_table_full_name,
  target_column_name
from system.access.column_lineage
where source_table_full_name = 'main.silver.customers'
  and source_column_name = 'email'
order by target_table_full_name;
```

**Common mistake.** Trusting stale lineage. If the crawler runs nightly, a table created this morning has no edges yet and the impact list is incomplete. State the lineage freshness whenever you present an impact analysis.

## What is the difference between governance and security?

Security focuses on preventing unauthorized access and ensuring confidentiality/integrity (access controls, encryption). Governance includes security but also covers ownership, definitions, quality, lifecycle, and change management. Governance answers "who owns this data and what does it mean", not just "who can access it".

| Concern | Security | Governance |
|---|---|---|
| Primary question | who can access it? | who owns it, what does it mean, how does it change? |
| Controls | IAM, encryption, network | catalog, glossary, contracts, retention |
| Failure mode | breach | wrong numbers, untraceable data, compliance gaps |
| Owner | security/CISO | data owner plus platform team |
| Evidence | access logs, pen tests | lineage, contracts, access reviews |

They overlap at classification: you cannot secure sensitive data if nobody has classified it, which is why governance usually drives the tag that security then enforces.

**How they meet in the platform.** Classification is the handshake. Governance decides that a column is `pii`, who owns it, and how long it is kept. Security turns that decision into a grant, a mask, or a row filter. If classification is missing, security has nothing to enforce; if enforcement is missing, classification is only a label.

Worked example: the same email column raises two questions. Governance asks who owns it, how long it is kept, and whether the definition is agreed. Security asks which roles can query it and what those roles see. The second question cannot be answered well without the first.

**Common mistake.** Letting the security team classify data alone. Stewards know what a field means, so they should propose the tag, and security should own the enforcement mechanism.

## What are common roles in data governance (owner/steward/custodian)?

Common roles are:
+ data owner: accountable for the dataset and its business definition
+ data steward: maintains quality, metadata, and processes
+ data custodian: operates the technical platform (storage, access, backups)

Clear roles prevent "nobody owns it" incidents.

```text
Data owner      - business accountability (VP of Sales owns customer master).
Data steward    - day-to-day definitions, quality rules, dispute resolution.
Data custodian  - platform engineering: storage, pipelines, backups, access plumbing.
Data consumer   - analysts and data scientists who must follow the contract.
Privacy officer - legal/compliance for PII/PHI and erasure requests.
```

Interview note: the most common governance failure is an ownership gap at the boundary between teams, for example a table produced by a pipeline team but consumed as a business metric by finance. Name the accountable owner explicitly in the catalog.

**How the roles divide the work.** Ownership is accountability, stewardship is day-to-day care, custody is running the platform.

| Role | Accountable for | Typical task |
|---|---|---|
| Data owner | the dataset and its meaning | approves a breaking change |
| Data steward | quality, definitions, classification | fixes a mislabelled column |
| Data custodian | storage, backups, access plumbing | applies the grant |

Worked example: a new `loyalty_tier` column lands. The steward classifies and documents it, the custodian adds the tag to the catalog and applies the grant, and the owner signs off that the business definition matches finance's.

**Common mistake.** Naming an owner who has no authority over the producer. Ownership without the power to approve or block a change is just a name in a table.

## What is a data catalog and what should it contain?

A data catalog is an inventory of datasets with searchable metadata. It typically contains dataset descriptions, owners, lineage, classifications, freshness/SLA, schema, and links to dashboards. A good catalog reduces time to discovery and improves trust and reuse.

Minimum useful catalog fields:

```yaml
asset: main.gold.fact_orders
kind: table
description: "One row per completed order, conformed for finance reporting."
owner: orders-platform@example.com
steward: finance-data@example.com
domain: sales
classification: [confidential, pii-adjacent]
tags: [tier-1, finance-certified]
freshness_sla_minutes: 30
retention_days: 2555
schema:
  - {name: order_id, type: string, pii: false}
  - {name: customer_token, type: string, pii: true, policy: tokenized}
links:
  runbook: https://wiki.example.com/runbooks/fact-orders
  dashboard: https://bi.example.com/dashboards/revenue
  source_of_truth: orders-db.public.orders
```

Certification is what turns a catalog into a trust signal: mark tier-1/finance datasets as certified only after contract tests, ownership, and freshness SLAs are in place.

**How harvesting works in practice.** A crawler or a push integration reads table and column metadata, statistics, and tags, then writes them into the catalog on a schedule. The catalog also stores operational metadata: last refresh, query counts, downstream usage, and freshness. Usage metadata is what makes a catalog searchable by relevance rather than only by name, and it is what separates a live catalog from a stale wiki.

The minimum useful set is small: asset name, physical location, owner, description, schema, tags, freshness, and certification status. Everything else is an enrichment.

Worked example: find tables that exist in the warehouse but have not been documented yet, which is the backlog of ungoverned data.

```sql
select
  table_schema,
  table_name,
  table_type
from main.information_schema.tables
where table_schema not in ('information_schema', 'security')
order by table_schema, table_name;
```

```json
{
  "asset": "main.gold.fact_orders",
  "owner": "orders-platform@example.com",
  "steward": "finance-steward@example.com",
  "certified": true,
  "tier": 1,
  "certified_by": "data-council",
  "review_by": "2026-12-31"
}
```

**Common mistake.** Cataloguing only the tables someone asked about. Coverage should be measured as a percentage of tier-1 assets, and new tables should be discovered automatically.

## What is business glossary vs technical metadata?

A business glossary defines business terms (for example "active user", "net revenue") and their approved definitions. Technical metadata describes datasets and fields (schemas, types, partitions, lineage). Both are needed: glossary aligns meaning, technical metadata enables implementation and discovery.

```text
Business glossary item:
  term: net_revenue
  definition: gross_revenue - refunds - chargebacks, excluding tax
  owner: finance-data@example.com
  synonyms: [net sales, net bookings]
  related_assets: [gold.fact_orders, gold.revenue_daily]
```


Interview point: glossary without technical metadata is documentation that drifts; technical metadata without a glossary is a schema nobody agrees on. Link each glossary term to the certified asset and column that implements it.

**How the two link.** A glossary term has an approved definition and an owner. Technical metadata describes physical objects. The link is a mapping: the term `net_revenue` points at the metric definition, the model that computes it, and the columns that implement it. With the mapping you can test the definition against real data; without it the glossary is prose.

```yaml
term: net_revenue
owner: finance-analytics@example.com
definition: gross_revenue - refunds - chargebacks, excluding internal test orders
mapped_assets:
  - main.gold.fact_orders.amount
  - main.gold.fact_refunds.amount
synonyms: [net sales, net bookings]
```

**Common mistake.** Two glossaries. When finance and marketing each own a definition of "active user", the catalog becomes an argument instead of a reference. One term, one owner.

## RBAC vs ABAC: what is the difference?

RBAC (Role-Based Access Control) grants permissions to roles (analyst, engineer) and assigns users to roles. ABAC (Attribute-Based Access Control) uses attributes (department, region, sensitivity) to evaluate policies dynamically. ABAC is more flexible but can be harder to manage without good tooling.

```text
RBAC:
  grant role analyst to user alice;
  grant select on table gold.fact_orders to role analyst;
```


| Dimension | RBAC | ABAC |
|---|---|---|
| Granularity | table/schema | row, column, attribute |
| Scale | many roles, role explosion | policy count stays small |
| Audit | simple to explain | needs policy evaluation logs |
| Best for | coarse access, small orgs | multi-tenant, regional, regulated data |

Most modern lakehouses combine them: RBAC grants the base privilege, ABAC (row filters, column masks, tag policies) refines it.

**How the engine evaluates a request.** In an RBAC-only system the engine maps the user to roles, unions the privileges of those roles, and checks the requested object. In an ABAC system the engine also resolves attributes: the caller's region or clearance, and the resource's tags. Row filters and column masks are the enforcement form of ABAC, because they change the data returned rather than only allowing or denying the query.

**Purpose-based access** is ABAC with the purpose as one more attribute. The caller declares or inherits a purpose such as `treatment` or `billing`, and the policy compares it with the tag on the data.

Worked example: an analyst holds `select` on `gold.fact_orders` through a role. The table also carries `region=APAC`. A row filter compares the row's region with the caller attribute, and a mask hides `card_number` unless the caller's purpose is `billing`.

**Common mistake.** Treating a role grant as sufficient. `select` on a table with no attached policy means every row and every column is visible; the grant is only the base privilege.

## What is row-level and column-level security?

Row-level security restricts which rows a user can see (for example only their region). Column-level security restricts which columns are visible (for example hide salary or PII). These controls are important for multi-tenant analytics and compliance.

```text
Row-level: a regional manager in APAC sees only APAC orders, even though the table
           contains global orders.
Column-level: an analyst can see customer_id but sees a masked email.
Cell-level: the combination, enforced by a row filter plus a column mask.
```

Both controls should be enforced in the query engine so that BI tools, notebooks, and ad-hoc SQL all inherit the same policy. Enforcing only in a BI tool's semantic layer leaves a hole for anyone with direct SQL access.

**How row filters and column masks execute.** A row filter is a function that returns a boolean per row; the engine injects it as an extra predicate, so the plan changes and partition pruning can suffer. A column mask is a function that returns the value the caller is allowed to see. Both usually run as the definer rather than as the caller, which is what stops a user from bypassing the logic.

**Where it can leak.** A filter is only as strong as the paths to the data. Aggregations over the filtered table are fine, but a join to an unfiltered copy of the same rows, a materialized clone, or an export that runs with a stronger role can all defeat it.

**Common mistake.** Assuming a filter covers derived tables. Copies and extracts created before the policy was attached do not inherit it, so attach policies at the schema level and scan for copies.

## How do you implement row-level and column-level security in practice?

Unity Catalog (Databricks) example: a row filter function plus a column mask function, attached to a table.

```sql
-- Row filter: only rows for the caller's region.
create or replace function main.security.region_filter(region string)
returns boolean
return region = current_user_region();

alter table main.gold.fact_orders
set row filter main.security.region_filter on (region);

-- Column mask: show the last four digits unless the caller is in the PII group.
create or replace function main.security.mask_email(email string)
returns string
return case
  when is_account_group_member('pii_readers') then email
  else concat('***', right(email, 4))
end;

alter table main.silver.customers
alter column email set mask main.security.mask_email;
```

Snowflake example:

```sql
create or replace masking policy mask_email as (val string) returns string ->
  case
    when current_role() in ('PII_READERS') then val
    else regexp_replace(val, '(^[^@]+)@', '****@')
  end;

alter table silver.customers modify column email set masking policy mask_email;

create or replace row access policy rap_region as (region string) returns boolean ->
  region = current_role();

alter table gold.fact_orders add row access policy rap_region on (region);
```

BigQuery uses policy tags for column-level security:

```bash
bq query --use_legacy_sql=false \
  'CREATE OR REPLACE TABLE gold.fact_orders (
     order_id STRING,
     customer_email STRING OPTIONS(policy_tags=["projects/p/locations/us/taxonomies/1/policyTags/2"]),
     amount NUMERIC
   )'
```

Interview guidance: state how you test these controls (a CI test that assumes the role and asserts the filtered count) and how you audit changes (who altered a policy and when).

**Test the control, not only the policy text.** The reliable test assumes the restricted identity and asserts the result.

```python
# Review-only sketch of a control test.
def assert_hidden(rows, column, forbidden):
    for row in rows:
        assert row[column] not in forbidden, f"leaked {forbidden!r} in {row}"
```

```bash
# Snowflake: run the query as the restricted role and inspect the result.
snowsql -q "use role analyst_apac; select region, count(*) from gold.fact_orders group by 1;"
```

Worked example: a fintech attaches `region_filter` to every fact table in the `gold` schema, then runs one CI job per role. The job fails if an APAC analyst sees a non-APAC region, or if a masked value appears in clear text.

**Common mistake.** Exposing a masked table through a view that was built with much stronger definer rights. The view can re-expose exactly the column the mask was hiding.

## How do you handle PII/PHI data in analytics platforms?

You classify sensitive fields, minimize exposure, and enforce least privilege. Common controls include encryption, masking/tokenization, column-level access, and auditing. You also ensure downstream marts only include necessary fields and implement secure sharing patterns.

A practical layered approach:

```text
1. Discover and classify: scan for emails, national IDs, card numbers, health codes.
2. Minimize: do not land raw PII in the lake at all; tokenize at the app boundary.
3. Isolate: keep a short-retention vault table with raw PII, exempt from broad grants.
4. Protect: encryption at rest/in transit, column masks, row filters, tag-based policies.
5. Govern: retention, erasure, access reviews, and audit logs on all of the above.
6. Propagate: column-level lineage flags derived columns that still carry PII.
```

PHI adds HIPAA-style requirements on top: minimum necessary access, business associate agreements, breach notification, and stricter audit retention. In an interview, mention that PHI often requires its own catalog classification and a separate access-approval workflow rather than a self-service request.

**Discovery mechanics.** A scanner samples rows, applies recognizers (regular expressions, checksums such as the Luhn test for card numbers, dictionaries, and language models), and emits a candidate tag with a confidence and a few matched samples. A human or a second signal confirms before the tag becomes governed, because a false positive that masks a needed field breaks analytics.

**PHI adds scope.** Protected health information is personal data in a health context, so the same column can be PII in one table and PHI in another. Under HIPAA the operating rule is minimum necessary: grant the narrowest access that still lets the role do its job.

**Common mistake.** Scanning only structured columns. PII hides in free-text support tickets, JSON payloads, and log lines, so sampling must include semi-structured fields.

A recognizer is a detector plus a decision rule. This review-only sketch keeps the two separate so the decision can be tested.

```python
import re

EMAIL = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+[.][A-Za-z]{2,}")


def match_rate(samples, pattern):
    hits = [value for value in samples if pattern.search(str(value))]
    return len(hits), len(samples), len(hits) / max(len(samples), 1)


hits, total, rate = match_rate(["a@b.com", "not an email", "c@d.org"], EMAIL)
print(hits, total, round(rate, 2))
```

## What is data masking and tokenization?

Masking hides sensitive values (for example showing only last 4 digits) while keeping the dataset usable. Tokenization replaces sensitive values with tokens that can be detokenized only with access to a secure mapping. Tokenization is useful when analytics needs stable identifiers without exposing raw PII.

Comparison:

| Technique | Reversible | Stable across rows | Use case |
|---|---|---|---|
| Static masking | no | n/a | non-production copies |
| Dynamic masking | no | n/a | role-based display in production |
| Hashing (unsalted) | no | yes | joins on a pseudonymous key |
| Salted hashing / HMAC | no | yes (with same salt) | pseudonymization for analytics |
| Tokenization | yes, with vault | yes | stable key plus controlled re-identification |
| Format-preserving encryption | yes, with key | yes | payment data where format must remain valid |

Deterministic tokenization in PySpark, using HMAC so the same input always maps to the same token:

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
import hmac
import hashlib
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

SALT = b"rotate-me-in-a-secret-manager"

def tokenize(value: str) -> str:
    if value is None:
        return None
    digest = hmac.new(SALT, value.encode("utf-8"), hashlib.sha256).hexdigest()
    return digest[:32]

tokenize_udf = F.udf(tokenize, StringType())

customers = spark.table("silver.customers").withColumn(
    "customer_token", tokenize_udf(F.col("email"))
)

Interview caution: unsalted `sha256(email)` is trivially reversible for a small email space via rainbow tables; always salt or HMAC. Also decide whether the token must survive a key rotation, because rotating the salt breaks historical joins.

**How a masking policy is evaluated.** The engine calls the policy function for each access to the column, passing the value and the session context. The function returns either the original value or a replacement. Because the policy is attached to the column, every reader inherits it, including BI tools and notebooks.

**Tokenization.** A token replaces a value with a surrogate that is stable for the same input, usually an HMAC with a secret key. The mapping that allows detokenization lives in a separate, tightly held store. Joins keep working on the token while the raw value never reaches the analytics layer.

Worked example: `card_number` gets a mask that shows the last four digits to support, while `customer_email` gets an HMAC token so the same customer joins across fact tables. Only the vault schema holds raw values, and its role is granted to a small group.

**Common mistake.** Masking in the BI layer only. A semantic-model expression hides the column on one dashboard, but anyone can still query the table directly. Enforce at the engine so every path behaves the same.

## How do you tag and classify data in a catalog?

Tags are the bridge between governance intent and enforcement. Classify columns with a small, controlled taxonomy rather than free-form strings.

```yaml
# classification taxonomy
classification:
  public:        {rank: 0}
  internal:      {rank: 1}
  confidential:  {rank: 2}
  restricted:    {rank: 3}   # PII, PHI, PCI, secrets
```


Unity Catalog tagging:

```sql
create catalog if not exists main;
create schema if not exists main.security;

alter table main.silver.customers
alter column email set tags ('pii_type' = 'email', 'classification' = 'restricted');

alter table main.gold.fact_orders set tags ('domain' = 'sales', 'tier' = '1');
```

Automate tagging from an information schema scan so new columns are classified within a day, and make unclassified `restricted`-looking columns fail a CI check.

**Tag governance.** A tag is useful when it is controlled. Governed tags are defined once at the account or organization level with a fixed set of allowed values, and only a small group may assign them. Ad-hoc string tags drift, so they cannot drive enforcement.

Tags also propagate. A tag on a table can set the default for its columns, and a lineage-aware scanner can copy a tag onto derived columns. Propagation is what keeps coverage high when a new model selects an already-classified column.

Worked example: measure which columns carry a governed tag before an audit.

```sql
-- Databricks: governed tags attached to columns.
select
  column_name,
  tag_name,
  tag_value
from system.information_schema.column_tags
where catalog_name = 'main'
  and schema_name = 'silver'
  and table_name = 'customers';
```

**Common mistake.** Allowing anyone to invent a tag. Two spellings of `pii` mean one of them is invisible to the policy engine, and the data it labels is unprotected.

## How do you implement tag-based access control in Unity Catalog?

Unity Catalog ABAC policies attach governed tags to data and let a policy grant or deny access based on those tags.

```sql
-- 1. Define governed tags at the account level (done once by an account admin).
--    Example tags: classification, pii_type, domain.

-- 2. Tag data.
alter table main.silver.customers
alter column email set tags ('pii_type' = 'email');

-- 3. Create a policy: only the pii_readers group may see restricted PII columns.
create policy main.security.pii_column_policy
on catalog main
comment 'Mask PII columns for everyone except pii_readers'
as
  for select
  to all
  except (pii_readers)
  match columns with tag pii_type
  using column mask main.security.mask_email;
```

```sql
-- Row-level ABAC by governed region tag.
create policy main.security.region_rows
on schema main.gold
as
  for select
  to all
  match tables with tag domain = 'sales'
  using row filter main.security.region_filter;
```

Interview points:

+ Tags must be governed (account-level) to be trustworthy; a schema-local tag anyone can set is not a security boundary.
+ Policy evaluation happens at query time, so ad-hoc SQL, notebooks, and BI all inherit it.
+ Audit `system.access.audit` to prove who saw or changed what.

**How the policy is evaluated.** A governed tag is attached to a catalog, schema, table, or column. A policy names the tag and one or more principals, then grants or denies an action on every object that carries that tag. Because the policy matches tags rather than object names, a new table inherits the rule as soon as it is tagged, which is the whole point of ABAC.

**Common mistake.** Tagging after the fact. A table that holds sensitive data but carries no tag matches no policy, and the default grant still applies. Tag at creation time in the pipeline, and add a daily job that lists unclassified columns.

## How do you implement Lake Formation tag-based access control?

AWS Lake Formation uses LF-Tags to express ABAC: you tag data assets and grant permissions to principals by tag expression instead of per-table grants.

```yaml
# Terraform-style sketch of LF-Tag based access control.
resource "aws_lakeformation_lf_tag" "classification" {
  key    = "classification"
  values = ["public", "internal", "confidential", "restricted"]
}
```


Lake Formation also supports row-level and cell-level security by attaching a data filter expression and column lists to a grant. Inspect effective permissions with `aws lakeformation list-permissions` and monitor access through CloudTrail.

**How LF-Tags work.** An LF-Tag is a key with allowed values, for example `classification` with values `public`, `confidential`, and `restricted`. You attach tags to a database, table, or column, then grant permissions to principals by tag expression rather than by resource name. A resource that is tagged later is covered by the existing grant immediately.

Worked example: tag the `raw` database `classification=restricted` and grant `analyst` select only where `classification=public`. Every current and future public table in `raw` is readable by analysts, and nobody has to enumerate table names.

**Common mistake.** Forgetting that cell-level security is a separate feature from LF-Tag grants. The tag grant controls who may read; the data filter controls which rows and columns come back. Most restricted tables need both.

## What is policy as code for data governance?

Policy as code means governance rules live in version control, are reviewed like software, and are applied by automation instead of a wiki and good intentions.

```yaml
# policies/pii_access.yaml
apiVersion: governance.example.com/v1
kind: DataPolicy
metadata:
  name: pii-read-access
spec:
  match:
    tags: {pii_type: "*"}
  rules:
    - effect: deny
      principals: {not_in_group: pii_readers}
      actions: [select, export, download]
    - effect: allow
      principals: {in_group: pii_readers}
      actions: [select]
      conditions:
        - mfa_required: true
        - purpose: [fraud_investigation, compliance]
  exceptions:
    process: access-request-ticket
    max_duration_days: 30
```

Benefits: reviewable diffs, testable policy (unit tests assert allow/deny decisions), consistent enforcement across engines, and an audit trail linking a policy version to a change window.

In [ ]:
# Unit test a policy decision before deploying it.
def evaluate(subject, action, resource_tags, policy):
    if "pii_type" in resource_tags and "pii_readers" not in subject["groups"]:
        return "deny"
    return "allow"

assert evaluate({"groups": []}, "select", {"pii_type": "email"}, {}) == "deny"
assert evaluate({"groups": ["pii_readers"]}, "select", {"pii_type": "email"}, {}) == "allow"

**How policy as code is deployed.** Policies live in git beside the pipeline code. A pull request runs unit tests that assert allow and deny decisions, then a plan step that shows the diff, then an apply step with a service identity that can write only policy objects. The same pipeline publishes the rules to the catalog so people can read them.

```yaml
# .github/workflows/governance.yml
name: governance
on: [pull_request]
jobs:
  policy:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: unit test policies
        run: python policies/tests/test_decisions.py
      - name: plan
        run: ./bin/policy plan --dir policies/
```

Worked example: a steward opens a pull request that adds `pii` masking to `silver.customers.phone`. CI runs the decision tests, the plan shows one new masking policy and one altered column, and a second reviewer approves. The change is recorded with an author, a reason, and a timestamp, which is exactly the evidence an auditor asks for.

**Common mistake.** Letting console changes bypass the pipeline. If policies can be edited by hand, the repository stops being the source of truth. Detect drift with a scheduled job that compares live policy with committed policy.

## How do you implement auditability for data access?

You log access events (who queried what, when, from where) and retain logs per policy. You also track permission changes and administrative actions. Audit logs should be queryable for investigations and compliance reporting.

```sql
-- Unity Catalog audit events: reads, writes, and permission changes.
select
  event_time,
  user_identity.email as actor,
  action_name,
  request_params.full_name_arg as table_name,
  source_ip_address,
  response.status_code
from system.access.audit
where action_name in ('getTable', 'commandSubmit', 'updatePermissions', 'createTable')
  and event_time > current_timestamp() - interval 7 days
order by event_time desc;
```

```sql
-- Snowflake access history for a sensitive table, flattened to object names.
select
  h.query_start_time,
  h.user_name,
  o.value:objectName::string as object_name
from snowflake.account_usage.access_history as h,
     lateral flatten(input => h.direct_objects_accessed) as o
where o.value:objectName::string = 'SILVER.CUSTOMERS'
order by h.query_start_time desc
limit 100;
```

What a good audit story covers: access events, export/download events, permission changes, policy changes, retention/erasure actions, and admin actions on the account. Protect the audit log itself with restricted access and immutable storage.

**How access events become evidence.** The platform writes access events to an audit table or log sink: principal, action, object, timestamp, source address, and often the query text. A scheduled job copies them to immutable storage with a retention lock. Alerts fire on rare events such as permission grants, exports to a new location, or access from an unfamiliar network.

```json
{
  "event_time": "2026-10-04T13:21:07Z",
  "principal": "analyst@example.com",
  "action": "SELECT",
  "object": "main.gold.fact_orders",
  "columns": ["order_id", "amount"],
  "row_count": 48211,
  "source_ip": "10.4.2.19"
}
```

Worked example: a quarterly review joins permission changes to access events. If a principal was granted a sensitive table and never queried it, the reviewer revokes the grant, which is the cheapest way to shrink the attack surface.

**Common mistake.** Retaining logs for less time than the policy review cycle, or storing them where the audited administrator can delete them. Evidence must outlive the period it covers and be write-once from the reviewer's point of view.

## How do you handle GDPR right to be forgotten in a lakehouse or warehouse?

You need a process to delete or anonymize personal data across raw, derived, and serving layers. This can involve targeted deletes (where supported), reprocessing from raw logs without the user, or storing personal data separately to allow deletion without rewriting large datasets. You also document what is feasible given retention and legal constraints.

```text
Erasure runbook:
1. Resolve the subject to every identifier (customer_id, email token, device id).
2. Vault: delete the raw PII mapping.
3. Raw/bronze: delete or expire affected partitions and compact so the data is gone.
4. Silver/gold: hard-delete keyed rows and rebuild aggregates that included them.
5. Downstream: purge search indexes, caches, exports, and ML feature stores.
6. Evidence: write an immutable erasure record (request id, scope, operator, timestamp).
7. Backups: rely on documented backup expiry; do not promise instant backup erasure.
```

```sql
-- Iceberg makes targeted deletes and snapshot expiry practical.
delete from main.bronze.events_raw where customer_token = :token;
delete from main.silver.customers_current where customer_token = :token;
delete from main.silver.customer_history where customer_token = :token;

-- Expire the snapshots that still reference the deleted rows.
call main.system.expire_snapshots(
  table => 'bronze.events_raw',
  older_than => timestamp '2026-10-04 00:00:00',
  retain_last => 1
);
```

Interview point: deleting a row from a table with time travel or an immutable raw log is not erasure until old snapshots and files are expired. State the retention guarantee explicitly: "new queries cannot see it immediately; physical removal completes at the next compaction/expiry, within N days."

**How erasure propagates.** A request arrives for one data subject. You resolve the subject to every key it appears under, then use lineage to find every table that carries those keys: raw events, derived marts, feature tables, and caches. The job deletes or re-tokenizes each one, records a receipt per table, and expires historical snapshots so the old value is not resurrected.

```sql
-- Databricks: downstream tables that read the raw personal data.
select distinct target_table_full_name
from system.access.table_lineage
where source_table_full_name = 'main.bronze.customers_raw'
order by 1;
```

Worked example: a customer asks for deletion. The raw table is append-only, so `customer_id` is crypto-shredded: the per-subject key is destroyed and every token of that subject becomes unlinkable. Serving tables then delete or overwrite the row by primary key, and snapshot expiry removes the older copies.

**Common mistake.** Deleting only from the serving layer. The raw copy, the backup, and the time-travel snapshot still hold the value, so the erasure is not complete.

## What are data retention policies and how do you enforce them?

Retention policies define how long data is kept and when it must be deleted or archived. Enforcement can be implemented with partition expiry jobs, lifecycle rules on object storage, snapshot expiration, and access reviews. Retention must align with legal requirements and business needs.

```yaml
# retention matrix
retention:
  raw_events:        {days: 90, action: delete, storage_class: standard}
  pii_vault:         {days: 365, action: delete, owner: privacy@example.com}
  silver_current:    {days: 1825, action: keep, review: annual}
  gold_aggregates:   {days: 2555, action: keep, review: annual}
  audit_logs:        {days: 3650, action: archive_immutable}
  backups:           {days: 35, action: expire}
```

```sql
-- Partition expiry for a date-partitioned raw table.
alter table main.bronze.events_raw drop partition (event_date < '2026-07-01');

-- Or delete with Iceberg and then expire snapshots + orphans.
delete from main.bronze.events_raw where event_date < date '2026-07-01';
```

```bash
# S3 lifecycle rule for the raw landing zone.
aws s3api put-bucket-lifecycle-configuration \
  --bucket company-raw \
  --lifecycle-configuration file://lifecycle.json
```

Interview note: retention and erasure interact. A shorter raw retention reduces erasure surface but also shortens replay ability; state the trade-off and who owns it (usually the privacy officer plus the platform owner).

**How retention is enforced.** A policy names the data class, the retention period, and the action at the end of the period: delete, archive, or anonymize. Enforcement is a scheduled job that reads the policy and applies it partition by partition, plus storage-level lifecycle rules for files. A legal hold overrides deletion and must be recorded, because deleting data under hold is a serious failure.

```yaml
retention:
  - dataset: main.bronze.events_raw
    days: 90
    action: delete
    basis: storage_limitation
  - dataset: main.silver.orders
    days: 2555
    action: archive
    basis: tax_law
  - dataset: main.gold.customer_360
    days: 3650
    action: delete
    legal_hold: false
```

Worked example: the raw zone keeps 90 days, the curated zone 7 years for tax, and the profile dataset 2 years. The job runs daily, records which partitions it dropped, and alerts when a partition is older than its policy allows, because a silent failure leaves data that should be gone.

**Common mistake.** Writing the policy without a job that proves it ran. Retention needs a run record with counts, otherwise it is indistinguishable from an untested note.

## How do you measure data governance maturity?

Governance needs metrics, or it becomes an unenforceable policy document. Useful measures:

+ Coverage: percentage of tier-1 datasets with an owner, definition, and freshness SLA.
+ Classification: percentage of columns auto-classified, and mean time to classify a new column.
+ Access hygiene: orphan permissions, over-privileged service accounts, access reviews completed on time.
+ Contract health: CI contract checks passing, breaking-change incidents per quarter.
+ Lifecycle: datasets past retention without an exemption, erasure requests closed within SLA.
+ Trust: certified-asset count and incident count caused by unannounced schema changes.

```text
Level 1 - ad hoc: no catalog, access by ticket, definitions in chat.
Level 2 - documented: catalog and glossary exist, ownership assigned, reviews manual.
Level 3 - enforced: tags drive policies, contracts in CI, retention automated.
Level 4 - trusted: certified assets, column-level lineage, measured freshness/compliance SLAs.
Level 5 - self-improving: policy decisions measured, exceptions expire automatically, cost and risk optimized.
```

**Make each level measurable.** A maturity model is useful only when each level has a number attached.

| Level | Signal | Example metric |
|---|---|---|
| Reactive | tickets drive fixes | mean time to classify a new column |
| Managed | catalog, owners, SLAs | percent of tier-1 datasets with an owner |
| Enforced | policy in the engine | percent of sensitive columns with a mask |
| Optimized | policy as code, measured reviews | policy drift detected within a day |

Worked example: an organization reports 92 percent owner coverage for tier-1 datasets, 78 percent classification coverage, and a 2-day median time to classify. The gap between owner coverage and classification coverage is the backlog to work next, and it is a number rather than an opinion.

**Common mistake.** Reporting activity instead of outcome. The number of policies written says nothing about exposure. Report coverage and enforcement, and track the same figures over time so improvement is visible.

## How do you operationalize governance at scale?

At small scale governance is a person and a spreadsheet. At scale it is a pipeline: rules are versioned, classification is automated, lineage drives change review, and access is re-certified on a schedule. The four sections below cover the four operational loops that keep a large platform honest. Each loop has an owner, a cadence, and a measurable output.

The loops share data. Classification feeds the tags that policies match. Lineage feeds the change review and the erasure job. Audit events feed the access review. When one loop is missing, the others quietly degrade.

## How do you enforce policy as code in CI?

**The loop.** Define the rule in git. Test the decision in CI. Plan the change and review the diff. Apply with a scoped identity. Detect drift on a schedule. The pipeline is the control, because it makes every change reviewable and every reviewer accountable.

A decision test is a plain function call. Keeping the policy logic in a library lets the same code run in CI and in the enforcement service.

```python
def allowed(subject, action, resource):
    if resource["classification"] == "restricted" and not subject.get("break_glass"):
        return False
    return action == "read" and subject["team"] in resource["allowed_teams"]
```

```bash
# Fail the build when live policy differs from the committed policy.
bin/policy diff --dir policies/ --live || { echo "policy drift detected"; exit 1; }
```

Worked example: a team wants a temporary exception for a migration. The exception is a commit with an expiry date. The scheduled drift job fails once the expiry passes, which forces the exception to be removed instead of forgotten.

**Common mistake.** Encoding policy in a console and exporting it later. The console becomes the source of truth, the repository becomes a snapshot, and the first unreviewed change is the one that causes the incident.

## How do you automate classification and PII discovery?

**The pipeline.** Sample rows from every new or changed table, run recognizers over the samples, score each candidate column, and write proposals to a review queue. Confirmed proposals become governed tags. Store the sample size, the match count, and the confidence with each proposal, because the reviewer needs to judge the evidence and not only the label.

Two dials control the cost. Sample size controls how small a PII rate you can detect. Confidence controls how often you are willing to be wrong. Both are arithmetic, not guesswork, which is why this loop can be tested.

A recognizer is a detector plus a decision rule. Card numbers pass a Luhn check, which removes most random digit strings.

```python
import re

CARD = re.compile(r"\b(?:\d[ -]?){13,16}\b")


def luhn(number):
    digits = [int(d) for d in re.sub(r"\D", "", number)]
    checksum = 0
    for index, digit in enumerate(reversed(digits)):
        if index % 2 == 1:
            digit *= 2
            if digit > 9:
                digit -= 9
        checksum += digit
    return checksum % 10 == 0
```

Worked example: a scan of 5,000 sampled support tickets finds 312 email matches in one column and 4 card matches in another. The email column becomes a confirmed `pii` tag. The four card matches go to a human, because a sample that small is weak evidence.

**Common mistake.** Auto-applying every high-confidence proposal. Confidence is a property of the detector, not of the column's meaning, and a column of order numbers can look like a national ID. Review false positives before the tag becomes governed, and measure the false-positive rate so the queue stays trustworthy.

## How do you use lineage for change management and impact analysis?

**The loop.** A producer proposes a change. The lineage service finds every downstream asset within N hops, the ownership registry names who to notify, and the change is blocked until the affected owners acknowledge it or the change is made compatible. After the deploy, the same edges tell you what to watch.

Not every downstream asset deserves the same attention. Score each one by hops from the source and by whether it is certified or tier-1. A tier-1 dashboard is notified first; an unused sandbox table can be ignored.

Worked example: a column type changes from `int` to `bigint`. Table-level lineage gives the blast radius and column-level lineage tells you which downstream columns actually read the changed field.

```sql
-- Databricks: tables that read the changed table, most recent first.
select
  target_table_full_name,
  max(event_time) as last_seen
from system.access.table_lineage
where source_table_full_name = 'main.silver.customers'
group by target_table_full_name
order by last_seen desc;
```

**Common mistake.** Reporting only the count of downstream tables. A blast radius of 300 tables means nothing without tier and usage. Join lineage to usage metadata, or the list gets ignored and the process dies.

## How do you run access reviews, certifications and audit evidence?

**The review loop.** On a cadence, list every principal, the entitlements they hold, and the last time they used each one. The manager or data owner certifies keep or revoke. The system records the decision, the reviewer, and the date, and removes revoked entitlements automatically. That same record is the audit evidence.

The useful input is usage. An entitlement that has not been exercised in 180 days is the easiest one to remove, and removing it lowers the risk that the next audit finds.

Worked example: a quarterly review of a sensitive dataset. The reviewer sees who can read it, when each principal last read it, and whether the grant comes from a role or a direct exception. Unused direct grants go first.

```sql
select
  grantee_name,
  privilege,
  granted_on,
  created
from main.information_schema.table_privileges
where table_schema = 'gold'
  and table_name = 'fact_orders'
order by grantee_name;
```

```json
{
  "campaign": "2026-Q4-sensitive-access",
  "reviewer": "data-council",
  "decisions": [
    {"principal": "analyst_apac", "entitlement": "gold.fact_orders", "decision": "keep", "reason": "regional reporting"},
    {"principal": "contractor_17", "entitlement": "gold.fact_orders", "decision": "revoke", "reason": "no access in 180 days"}
  ]
}
```

**Common mistake.** Certifying by bulk action. A review where every line is approved in one click produces evidence that nobody looked. Sample the bulk approvals and report the sample size, or the certification is theater.

## Data governance exercises

These exercises use only the Python standard library and model the governance ideas with small, runnable calculations. Each one names the task, gives a hint, states the expected result, and then prints or asserts the answer. Run them with the repository virtual environment: `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

The exercises reuse the units the platforms use. Classification coverage is a weighted ratio over columns, not over tables. Discovery sample size comes from the binomial rule for at least one hit. Erasure cost is dominated by scanning unless the subject keys are indexed. Lineage blast radius is a breadth-first search over dependency edges.

Treat the asserted numbers as a check on the formula rather than a prediction about a real platform. Real scans add variance from sampling, skew, and caching, so the value of each exercise is knowing which quantity to compute and which direction a change moves it.

### Try it 1. Compute classification coverage from sampled tables

**Task.** Given three tables with a column count and a classified-column count, compute the weighted coverage over all columns and the unweighted mean of per-table coverage. Report the worst table.

**Hint.** Weighted coverage is `total classified / total columns`. The unweighted mean averages the per-table ratios, so a small table counts as much as a large one.

**Expected result.** Weighted coverage is about `38.7` percent, the unweighted mean is about `47.4` percent, and the worst table is `events`.

In [ ]:
tables = [
    {"name": "customers", "columns": 120, "classified": 96},
    {"name": "orders", "columns": 85, "classified": 40},
    {"name": "events", "columns": 240, "classified": 36},
]

total_columns = sum(t["columns"] for t in tables)
total_classified = sum(t["classified"] for t in tables)
weighted = total_classified / total_columns
unweighted = sum(t["classified"] / t["columns"] for t in tables) / len(tables)
worst = min(tables, key=lambda t: t["classified"] / t["columns"])

for t in tables:
    print(f"{t['name']:<10} {t['classified']:>3}/{t['columns']:<3} = {t['classified'] / t['columns']:.1%}")
print(f"weighted coverage   = {weighted:.1%}")
print(f"unweighted coverage = {unweighted:.1%}")
print("worst table:", worst["name"])

assert total_columns == 445
assert round(weighted, 3) == 0.387
assert abs(unweighted - 0.4735) < 1e-3
assert worst["name"] == "events"

### Try it 2. Choose a sampling rate for PII discovery

**Task.** Write a function that returns the number of rows to sample so that a PII value appearing in a given fraction of rows is found with a target confidence. Compute it for 1 percent and 0.1 percent prevalence at 95 percent confidence.

**Hint.** The chance of missing it in one row is `1 - prevalence`. The chance of missing it in `n` rows is `(1 - prevalence) ** n`, so solve `(1 - prevalence) ** n = 1 - confidence` and round up.

**Expected result.** 1 percent prevalence needs `299` rows and 0.1 percent needs `2995` rows. A higher confidence always increases the sample.

In [ ]:
import math


def sample_size(prevalence, confidence):
    return math.ceil(math.log(1 - confidence) / math.log(1 - prevalence))


for rate in (0.01, 0.001):
    print(f"prevalence {rate:.1%} -> {sample_size(rate, 0.95)} rows for 95% confidence")

assert sample_size(0.01, 0.95) == 299
assert sample_size(0.001, 0.95) == 2995
assert sample_size(0.01, 0.99) > sample_size(0.01, 0.95)

### Try it 3. Plan a retention and erasure job

**Task.** Given a daily volume of 40 million rows, 90 days of retention, 500 erasure requests per day, 12 rows per subject, a scan rate of 50 million rows per second, and a delete rate of 5,000 rows per second, compute the rows expired per day, the scan hours spent on erasure per day, and the actual delete seconds.

**Hint.** A steady state expires one full daily partition each day. Each erasure request scans `daily_rows * retention_days` rows unless the subject keys are indexed.

**Expected result.** 40,000,000 rows expire per day, each erasure scan takes `72` seconds, and 500 requests cost `10.0` hours of scanning against `1.2` seconds of real deletes.

In [ ]:
daily_rows = 40_000_000
retention_days = 90
requests_per_day = 500
rows_per_subject = 12
scan_rows_per_second = 50_000_000
erase_rows_per_second = 5_000

expired_rows_per_day = daily_rows
scan_rows_per_request = daily_rows * retention_days
scan_seconds_per_request = scan_rows_per_request / scan_rows_per_second
scan_hours_per_day = requests_per_day * scan_seconds_per_request / 3600
erase_seconds_per_day = requests_per_day * rows_per_subject / erase_rows_per_second

print(f"rows expiring per day      = {expired_rows_per_day:,}")
print(f"scan per erasure request   = {scan_rows_per_request:,} rows ({scan_seconds_per_request:.0f}s)")
print(f"erasure scan per day       = {scan_hours_per_day:.1f} hours")
print(f"actual delete work per day = {erase_seconds_per_day:.1f} seconds")
print("verdict: index the subject keys before scanning full history")

assert expired_rows_per_day == 40_000_000
assert scan_seconds_per_request == 72.0
assert round(scan_hours_per_day, 1) == 10.0
assert erase_seconds_per_day == 1.2

### Try it 4. Size an access review

**Task.** Given 400 principals, 120 datasets, an average of 6 entitlements per principal, half a minute of review per entitlement, and a 10 percent sample re-checked at 3 minutes each, compute the total effort and whether two reviewers working two 8-hour days can finish.

**Hint.** Effort is `entitlements * minutes / 60`. Capacity is `reviewers * days * hours per day`. Compare the two totals, and keep the sampling cost separate so it can be scaled down.

**Expected result.** 2,400 entitlements take `20.0` hours to review plus `12.0` hours of sampling, which is exactly `32.0` hours against a capacity of `32` hours.

In [ ]:
principals = 400
datasets = 120
entitlements_per_principal = 6
minutes_per_entitlement = 0.5
sample_rate = 0.10
minutes_per_sample = 3.0
reviewers = 2
review_days = 2
hours_per_reviewer_day = 8

entitlements = principals * entitlements_per_principal
review_hours = entitlements * minutes_per_entitlement / 60
sample_hours = entitlements * sample_rate * minutes_per_sample / 60
total_hours = review_hours + sample_hours
capacity_hours = reviewers * review_days * hours_per_reviewer_day
fits = total_hours <= capacity_hours

print(f"entitlements to review = {entitlements:,} across {datasets} datasets")
print(f"base review            = {review_hours:.1f} hours")
print(f"sampling audit         = {sample_hours:.1f} hours")
print(f"total                  = {total_hours:.1f} hours")
print(f"reviewer capacity      = {capacity_hours:.1f} hours")
print("fits:", fits)

assert entitlements == 2400
assert total_hours == 32.0
assert capacity_hours == 32
assert fits is True

### Try it 5. Detect a contract violation from a schema diff

**Task.** Compare an expected schema with the actual schema and report added columns, removed columns, and changed types. Treat a removed column or a changed type as a breaking contract violation.

**Hint.** Use set difference for added and removed names, then intersect the names and compare the declared types. Sort every list so the diff is stable between runs.

**Expected result.** The diff is one added column `region`, no removals, and one type change on `amount`, so the contract is breaking.

In [ ]:
expected = {"order_id": "string", "amount": "decimal(18,2)", "status": "string"}
actual = {"order_id": "string", "amount": "double", "status": "string", "region": "string"}


def diff(expected, actual):
    added = sorted(set(actual) - set(expected))
    removed = sorted(set(expected) - set(actual))
    changed = sorted(name for name in set(expected) & set(actual) if expected[name] != actual[name])
    return {"added": added, "removed": removed, "type_changed": changed}


result = diff(expected, actual)
breaking = bool(result["removed"] or result["type_changed"])
print("diff:", result)
print("breaking:", breaking)

assert result["added"] == ["region"]
assert result["removed"] == []
assert result["type_changed"] == ["amount"]
assert breaking is True

### Try it 6. Compute lineage blast radius from dependency edges

**Task.** Given a list of directed edges from each asset to the assets that read it, find every asset reachable from `raw.orders`, its depth, and the number of downstream assets.

**Hint.** Run a breadth-first search from the source, recording depth when a node is first seen. Use a deque for the frontier and a dictionary for visited depths.

**Expected result.** The blast radius is `6` downstream assets, `dash.revenue` sits at depth `3`, and the maximum depth is `3`.

In [ ]:
from collections import deque

edges = [
    ("raw.orders", "stg.orders"),
    ("stg.orders", "fct.orders"),
    ("fct.orders", "dash.revenue"),
    ("fct.orders", "ml.features"),
    ("raw.orders", "stg.customers"),
    ("stg.customers", "dim.customers"),
    ("dim.customers", "dash.revenue"),
]


def blast_radius(edges, source):
    adjacency = {}
    for src, dst in edges:
        adjacency.setdefault(src, []).append(dst)
    seen = {source: 0}
    queue = deque([source])
    while queue:
        node = queue.popleft()
        for nxt in adjacency.get(node, []):
            if nxt not in seen:
                seen[nxt] = seen[node] + 1
                queue.append(nxt)
    seen.pop(source)
    return seen


radius = blast_radius(edges, "raw.orders")
print("downstream assets:", len(radius))
for name, depth in sorted(radius.items(), key=lambda item: (item[1], item[0])):
    print(f"  depth {depth}: {name}")
print("max depth:", max(radius.values()))

assert len(radius) == 6
assert radius["dash.revenue"] == 3
assert max(radius.values()) == 3

### Try it 7. Estimate masking overhead on a query

**Task.** A query scans 500 million rows in 120 seconds. Three columns are masked at 20 nanoseconds per row per column, and the mask blocks predicate pushdown so 15 percent of the baseline time is lost. Compute the total time and the overhead percentage.

**Hint.** Mask cost in seconds is `rows * columns * nanoseconds / 1e9`. Pruning loss is a fraction of the baseline. The overhead is the increase divided by the baseline.

**Expected result.** Masking adds `30` seconds and lost pruning adds `18` seconds, so the total is `168` seconds and the overhead is `40.0` percent.

In [ ]:
baseline_seconds = 120
rows_scanned = 500_000_000
masked_columns = 3
mask_cost_ns_per_row_column = 20
pruning_loss_fraction = 0.15

mask_seconds = rows_scanned * masked_columns * mask_cost_ns_per_row_column / 1e9
pruning_seconds = baseline_seconds * pruning_loss_fraction
total_seconds = baseline_seconds + mask_seconds + pruning_seconds
overhead_pct = (total_seconds - baseline_seconds) / baseline_seconds * 100

print(f"baseline        = {baseline_seconds:.0f}s")
print(f"mask evaluation = {mask_seconds:.0f}s")
print(f"lost pruning    = {pruning_seconds:.0f}s")
print(f"total           = {total_seconds:.0f}s")
print(f"overhead        = {overhead_pct:.1f}%")

assert mask_seconds == 30.0
assert round(overhead_pct, 1) == 40.0
assert total_seconds > baseline_seconds

### Try it 8. Score governance maturity

**Task.** Given weighted scores for ownership, classification, enforcement, and audit, compute the overall maturity score and map it to a level: optimized at 0.80 or above, enforced at 0.60, managed at 0.40, otherwise reactive.

**Hint.** Check that the weights sum to 1.0 first, then compute the weighted sum with `sum(weight * score)`. Use `if` and `elif` in descending order so the highest band wins.

**Expected result.** The weights sum to `1.00`, the score is `0.7675` (about 76.8 percent), and the level is `enforced`.

In [ ]:
dimensions = {
    "ownership": {"weight": 0.30, "score": 0.90},
    "classification": {"weight": 0.25, "score": 0.75},
    "enforcement": {"weight": 0.25, "score": 0.60},
    "audit": {"weight": 0.20, "score": 0.80},
}

score = sum(d["weight"] * d["score"] for d in dimensions.values())
weight_total = sum(d["weight"] for d in dimensions.values())

if score >= 0.80:
    level = "optimized"
elif score >= 0.60:
    level = "enforced"
elif score >= 0.40:
    level = "managed"
else:
    level = "reactive"

print(f"weight total   = {weight_total:.2f}")
print(f"maturity score = {score:.4f} ({score:.1%})")
print("level:", level)
print("next fix:", min(dimensions, key=lambda name: dimensions[name]["score"]))

assert round(weight_total, 2) == 1.0
assert round(score, 4) == 0.7675
assert level == "enforced"

## Real world use cases for Data Governance

The scenarios below put the pieces together. Read them for the pattern rather than for the product names: classify before you protect, enforce in the engine, keep ownership explicit, and prove each control with a query. Every case ends with why the design holds up in production.

### 1. E-commerce PII protection and GDPR

Business context: an online retailer with 30 million customers must let 200 analysts query orders while keeping email, phone, and address out of reach, and must honor ~500 erasure requests per month. Volume: 4 TB of order data, 40 million events/day, 8 source systems.

Architecture: raw PII lives in a short-retention vault; the lakehouse stores `customer_token`; columns are tagged `pii_type`; a Unity Catalog policy masks email for everyone except the `pii_readers` group; an erasure orchestrator deletes tokens from bronze/silver and writes an immutable audit record.

```sql
alter table main.silver.customers
alter column email set tags ('pii_type' = 'email', 'classification' = 'restricted');

create policy main.security.pii_email
on catalog main
as
  for select
  to all
  except (pii_readers)
  match columns with tag pii_type
  using column mask main.security.mask_email;
```

Why this design: tokenization at the application boundary removes raw PII from the lake entirely, so erasure is a keyed delete plus a vault operation rather than a rewrite of immutable files, and tag-driven policy means new columns inherit protection without a per-table ticket.

### 2. Fintech multi-tenant analytics with regional isolation

Business context: a payments provider serves merchants in the EU, US, and APAC from one lakehouse and must guarantee that a merchant analyst never sees another region's data and that card data is never exposed. Volume: 50 million transactions/day, 1.2 PB total, 400 tenants.

Architecture: every table carries `region` and `tenant_id`; row access policies enforce region, row filters enforce tenant, and PCI columns are masked with tokenized values; governed tags drive the policies so a new table in the schema is covered automatically.

```sql
create or replace function main.security.region_filter(region string)
returns boolean
return region = current_user_region();

alter table main.gold.transactions
set row filter main.security.region_filter on (region);

create or replace function main.security.tenant_filter(tenant_id string)
returns boolean
return tenant_id = current_user_tenant();

alter table main.gold.transactions
set row filter main.security.tenant_filter on (tenant_id);
```

Why this design: cell-level access is enforced in the engine, so notebooks, BI, and ad-hoc SQL all behave identically; tag-driven policies scale to thousands of tables without thousands of grants; and the two filters compose so tenant isolation holds even inside a shared region.

### 3. Multi-tenant SaaS data contracts in CI

Business context: a SaaS platform feeds customer-facing analytics from 2,000 tenant schemas; an unannounced upstream rename previously broke dashboards for three days. Volume: 300 million events/day, 2,000 active contracts.

Architecture: contracts live in git, are validated against the produced tables in CI, and publish to the catalog on merge; a schema registry enforces `BACKWARD` compatibility on streaming topics; contract violations block the deploy and page the producing team.

```yaml
dataset: gold.tenant_events
owner: platform-analytics@example.com
schema:
  - {name: tenant_id, type: string, required: true}
  - {name: event_name, type: string, required: true}
  - {name: occurred_at, type: timestamp, required: true}
sla: {freshness_minutes: 15}
change_policy: {breaking: requires_new_version_and_consumer_signoff}
```

Why this design: moving the agreement from a wiki into CI turns "please tell us before renaming" into a build failure, and the catalog entry means every consumer can find the owner and the contract version without asking around.

### 4. Healthcare PHI lakehouse with audit and retention

Business context: a hospital network must let researchers query de-identified records while keeping PHI access limited, auditable, and retention-bound under HIPAA. Volume: 900 million clinical rows, 15 TB, 7-year audit retention.

Architecture: PHI is separated into a restricted schema with its own access workflow; researchers query a de-identified view with date shifting and removed identifiers; every access is logged; retention jobs expire partitions and snapshots on schedule.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
# De-identification view logic: shift dates and drop direct identifiers.
from pyspark.sql import functions as F

deidentified = (
    spark.table("restricted.encounters")
    .select(
        F.sha2(F.col("patient_mrn"), 256).alias("patient_token"),
        F.date_sub(F.col("admit_date"), 365).alias("admit_date_shifted"),
        F.col("diagnosis_code"),
        F.col("facility_id"),
    )
)

Why this design: researchers never touch the restricted schema, date shifting prevents re-identification by cross-referencing public records, and the immutable audit log plus retention automation is what satisfies the compliance review rather than a policy PDF.

### 5. Marketplace catalog and lineage-driven deprecation

Business context: an acquired company's data must be merged into the parent catalog, and 300 legacy tables must be deprecated without breaking reports. Volume: 12,000 tables, 400,000 columns, 5,000 dashboards.

Architecture: automated profiling classifies and describes every table; column-level lineage maps dashboard fields to source columns; each legacy table gets an owner, a deprecation date, and a redirect to its replacement; a weekly report shows remaining consumers.

```sql
select
  t.table_name,
  count(distinct l.target_table_full_name) as downstream_assets,
  max(l.event_time) as last_used
from system.information_schema.tables t
left join system.access.column_lineage l
  on l.source_table_full_name = concat_ws('.', t.table_catalog, t.table_schema, t.table_name)
where t.table_schema = 'legacy'
group by t.table_name
order by downstream_assets desc;
```

Why this design: lineage turns deprecation from guesswork into evidence, ownership makes someone accountable for each retirement, and the weekly consumer report gives holdouts a deadline instead of an accidental outage.